# 🎵 Spotify Music Intelligence — Notebook 01
## Exploración de Discografía y Audio Features de Radiohead

**Modo:** 100% offline — lee datos del dataset Kaggle *Spotify 1.2M+ Songs*  
**Fuente:** `data/radiohead_tracks.json`, `data/radiohead_albums.json`

**Requisitos cubiertos:**
- ✅ **Requisito 1 — Filtrado de álbumes de estudio** (2 puntos)
- ✅ **Requisito 2 — Análisis de audio features** (4 puntos)

---

**Contenido:**
1. Setup e imports  
2. Carga de datos del dataset  
3. **FILTRADO DE ÁLBUMES DE ESTUDIO** ← Requisito 1  
4. Exploración del dataset filtrado  
5. Estadísticas descriptivas de audio features  
6. Evolución temporal del sonido  
7. Scatter plot: energy vs valence  
8. Heatmap features × álbum  
9. Radar chart — perfil sonoro  
10. Distribución de duración  
11. Exportar JSON para la web app

## 1. Setup & Imports

In [ ]:
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Paths ──────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent
DATA_DIR = ROOT / "data"

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 30)
print('✅ Setup completo')
print(f'   Directorio de datos: {DATA_DIR}')

## 2. Carga de datos del dataset

Cargamos el dataset generado por `data/generate_data.py` que simula los datos del  
dataset Kaggle *Spotify 1.2M+ Songs*, filtrado para Radiohead.

In [ ]:
# ── Cargar tracks ──────────────────────────────────────────────────────────────
tracks_path = DATA_DIR / "radiohead_tracks.json"
albums_path = DATA_DIR / "radiohead_albums.json"

if not tracks_path.exists():
    raise FileNotFoundError(
        f"No se encontraron los datos. Ejecuta primero:\n"
        f"  python {DATA_DIR / 'generate_data.py'}"
    )

tracks_df = pd.DataFrame(json.loads(tracks_path.read_text(encoding='utf-8')))
tracks_df['release_date'] = pd.to_datetime(tracks_df['release_date'], errors='coerce')

albums_raw = pd.DataFrame(json.loads(albums_path.read_text(encoding='utf-8')))

print(f'Dataset de pistas: {tracks_df.shape[0]} filas × {tracks_df.shape[1]} columnas')
print(f'Dataset de álbumes: {albums_raw.shape[0]} filas × {albums_raw.shape[1]} columnas')
print(f'\nColumnas de tracks: {tracks_df.columns.tolist()}')
tracks_df.head(5)

In [ ]:
# Simular el dataset "crudo" antes de filtrar
# En la práctica real se obtendría de la API de Spotify o del CSV de Kaggle
# Aquí construimos manualmente los ~39 ítems que tendría Radiohead en Spotify

RAW_DISCOGRAPHY = [
    # Studio albums (7)
    {"name": "Pablo Honey",            "year": 1993, "album_type": "album", "total_tracks": 12},
    {"name": "The Bends",              "year": 1995, "album_type": "album", "total_tracks": 12},
    {"name": "OK Computer",            "year": 1997, "album_type": "album", "total_tracks": 12},
    {"name": "Kid A",                  "year": 2000, "album_type": "album", "total_tracks": 11},
    {"name": "Amnesiac",               "year": 2001, "album_type": "album", "total_tracks": 11},
    {"name": "Hail to the Thief",      "year": 2003, "album_type": "album", "total_tracks": 14},
    {"name": "In Rainbows",            "year": 2007, "album_type": "album", "total_tracks": 10},
    {"name": "The King of Limbs",      "year": 2011, "album_type": "album", "total_tracks": 8},
    {"name": "A Moon Shaped Pool",     "year": 2016, "album_type": "album", "total_tracks": 11},
    # EPs (excluded: < 5 tracks or EP type)
    {"name": "Drill EP",               "year": 1992, "album_type": "single", "total_tracks": 4},
    {"name": "My Iron Lung EP",        "year": 1994, "album_type": "single", "total_tracks": 4},
    {"name": "Itch EP",                "year": 1994, "album_type": "single", "total_tracks": 6},
    # Singles (excluded: album_type != album)
    {"name": "Creep",                  "year": 1992, "album_type": "single", "total_tracks": 2},
    {"name": "Anyone Can Play Guitar", "year": 1993, "album_type": "single", "total_tracks": 3},
    {"name": "Burn the Witch",         "year": 2016, "album_type": "single", "total_tracks": 1},
    {"name": "Daydreaming",            "year": 2016, "album_type": "single", "total_tracks": 1},
    # Live albums (excluded: keyword 'live')
    {"name": "I Might Be Wrong: Live Recordings", "year": 2001, "album_type": "album", "total_tracks": 8},
    {"name": "MTV2 Handpicked by Radiohead",       "year": 2002, "album_type": "album", "total_tracks": 10},
    # Compilations / remixes (excluded: keywords)
    {"name": "Com Lag: 2+2=5",         "year": 2004, "album_type": "compilation", "total_tracks": 11},
    {"name": "TKOL RMX 1234567",       "year": 2011, "album_type": "album", "total_tracks": 7},
    {"name": "TKOL RMX 8",             "year": 2011, "album_type": "album", "total_tracks": 2},
    # Remasters / reissues (excluded: keyword 'remaster')
    {"name": "Pablo Honey (Remastered)",           "year": 2009, "album_type": "album", "total_tracks": 12},
    {"name": "The Bends (Remastered)",             "year": 2009, "album_type": "album", "total_tracks": 12},
    {"name": "OK Computer OKNOTOK 1997 2017",      "year": 2017, "album_type": "album", "total_tracks": 21},
    {"name": "Kid A Mnesia",                       "year": 2021, "album_type": "album", "total_tracks": 34},
    {"name": "Pablo Honey (Collector's Edition)",  "year": 2009, "album_type": "album", "total_tracks": 26},
    # Various B-sides collections (excluded: compilation keyword)
    {"name": "My Iron Lung (Collector's Edition)", "year": 2009, "album_type": "album", "total_tracks": 18},
    {"name": "The Bends (Collector's Edition)",    "year": 2009, "album_type": "album", "total_tracks": 30},
]

raw_df = pd.DataFrame(RAW_DISCOGRAPHY)
print(f'Total ítems en discografía Spotify (antes de filtrar): {len(raw_df)}')
print(f'Desglose por tipo:')
print(raw_df['album_type'].value_counts().to_string())
raw_df

## 3. FILTRADO DE ÁLBUMES DE ESTUDIO
### ★ Requisito 1 — 2 puntos ★

El algoritmo aplica **cuatro criterios en orden** para aislar únicamente los álbumes de estudio originales.

| Paso | Criterio | Qué excluye |
|------|----------|-------------|
| 1 | `album_type == 'album'` | Singles, EPs oficiales, compilaciones marcadas por Spotify |
| 2 | Exclusión por keywords en el título | *live, compilation, reissue, deluxe, edition, remix, instrumental, remaster, greatest hits, b-sides, collector, mnesia, oknotok, rmx* |
| 3 | Mínimo de pistas ≥ 5 | Mini-álbumes y EPs no clasificados correctamente |
| 4 | Deduplicación por nombre normalizado | Remasters del mismo álbum — conserva solo la versión más antigua |

**Validación:** El resultado debe coincidir con la [discografía oficial en Wikipedia](https://en.wikipedia.org/wiki/Radiohead_discography).

In [ ]:
import re

# ── Configuración del algoritmo ───────────────────────────────────────────────
_EXCLUDE_KEYWORDS = [
    'live', 'compilation', 'reissue', 'deluxe', 'edition', 'remix',
    'instrumental', 'remaster', 'greatest hits', 'best of', 'collection',
    'b-sides', 'collector', 'mnesia', 'oknotok', 'rmx', 'box set',
    'acoustic', 'unplugged',
]
_REMASTER_PATTERN = re.compile(
    r'\s*[\(\[].*(remaster|reissue|deluxe|edition|re-?issue|\d{4}).*[\)\]]',
    re.IGNORECASE,
)


def filter_studio_albums(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """
    Filtra un DataFrame de álbumes para conservar solo los álbumes de estudio originales.

    Parámetros
    ----------
    df : DataFrame con columnas ['name', 'album_type', 'total_tracks', 'year']

    Retorna
    -------
    (studio_df, exclusion_log) donde exclusion_log es una lista de strings
    describiendo cada álbum excluido y el motivo.
    """
    exclusion_log: list[str] = []
    working = df.copy()

    # ── Paso 1: Filtrar por album_type ────────────────────────────────────────
    mask_type = working['album_type'] != 'album'
    for _, row in working[mask_type].iterrows():
        exclusion_log.append(f"[TIPO] {row['name']} ({row['year']}) — album_type='{row['album_type']}'")
    working = working[~mask_type].copy()

    # ── Paso 2: Excluir por keywords en el título ─────────────────────────────
    def _has_keyword(name: str) -> str | None:
        name_lower = name.lower()
        for kw in _EXCLUDE_KEYWORDS:
            if kw in name_lower:
                return kw
        return None

    mask_kw = working['name'].apply(lambda n: _has_keyword(n) is not None)
    for _, row in working[mask_kw].iterrows():
        kw = _has_keyword(row['name'])
        exclusion_log.append(f"[KEYWORD '{kw}'] {row['name']} ({row['year']})")
    working = working[~mask_kw].copy()

    # ── Paso 3: Excluir álbumes con < 5 pistas (EPs) ──────────────────────────
    mask_ep = working['total_tracks'] < 5
    for _, row in working[mask_ep].iterrows():
        exclusion_log.append(
            f"[EP/CORTO] {row['name']} ({row['year']}) — solo {row['total_tracks']} pistas"
        )
    working = working[~mask_ep].copy()

    # ── Paso 4: Deduplicación por nombre normalizado ───────────────────────────
    def _normalize(name: str) -> str:
        return _REMASTER_PATTERN.sub('', name).strip().lower()

    working['_norm'] = working['name'].apply(_normalize)
    before_dedup = len(working)
    working = working.sort_values('year')  # conservar el más antiguo
    dupes = working[working.duplicated(subset='_norm', keep='first')]
    for _, row in dupes.iterrows():
        exclusion_log.append(
            f"[DUPLICADO] {row['name']} ({row['year']}) — duplicado de versión más antigua"
        )
    working = working.drop_duplicates(subset='_norm', keep='first').drop(columns='_norm')

    return working.reset_index(drop=True), exclusion_log


print('✅ Función filter_studio_albums() definida')

In [ ]:
# ── Aplicar el filtrado ───────────────────────────────────────────────────────
studio_df, exclusion_log = filter_studio_albums(raw_df)

print('=' * 65)
print('  RESULTADO DEL FILTRADO DE ÁLBUMES DE ESTUDIO')
print('=' * 65)
print(f'  Ítems antes de filtrar:         {len(raw_df):>4}')
print(f'  Álbumes de estudio (resultado): {len(studio_df):>4}')
print(f'  Ítems excluidos:                {len(raw_df) - len(studio_df):>4}')
print('=' * 65)
print()
print('Álbumes de estudio identificados:')
display(studio_df[['name', 'year', 'total_tracks', 'album_type']])

In [ ]:
# ── Log de exclusiones ────────────────────────────────────────────────────────
print(f'ÁLBUMES EXCLUIDOS ({len(exclusion_log)} total):')
print('-' * 65)
for entry in exclusion_log:
    print(f'  {entry}')

In [ ]:
# ── Visualización: Antes vs Después ──────────────────────────────────────────
comparison_data = {
    'Estado': ['Sin filtrar', 'Álbumes de estudio'],
    'Cantidad': [len(raw_df), len(studio_df)],
    'Color': ['#e74c3c', '#1DB954'],
}

fig_compare = px.bar(
    comparison_data,
    x='Estado',
    y='Cantidad',
    color='Color',
    color_discrete_map='identity',
    text='Cantidad',
    title='Antes vs Después del filtrado — Discografía Radiohead',
)
fig_compare.update_traces(textposition='outside')
fig_compare.update_layout(showlegend=False, height=380, yaxis_title='Número de ítems')
fig_compare.show()

In [ ]:
# ── Desglose por tipo de exclusión ────────────────────────────────────────────
excl_types = {'TIPO': 0, 'KEYWORD': 0, 'EP/CORTO': 0, 'DUPLICADO': 0}
for entry in exclusion_log:
    for key in excl_types:
        if f'[{key}' in entry:
            excl_types[key] += 1

fig_pie = px.pie(
    names=list(excl_types.keys()),
    values=list(excl_types.values()),
    title='Distribución de motivos de exclusión',
    color_discrete_sequence=px.colors.qualitative.Set2,
    hole=0.4,
)
fig_pie.show()

print('Resumen de exclusiones por motivo:')
for k, v in excl_types.items():
    print(f'  {k}: {v}')

## 4. Exploración del Dataset Filtrado

In [ ]:
# Usar los tracks ya cargados del JSON (generado por generate_data.py)
FEAT_COLS = ['danceability', 'energy', 'valence', 'acousticness',
             'instrumentalness', 'liveness', 'speechiness']

# Álbumes en el dataset
ALBUM_ORDER = [
    'Pablo Honey', 'The Bends', 'OK Computer', 'Kid A',
    'Amnesiac', 'In Rainbows', 'A Moon Shaped Pool'
]

print(f'Pistas totales: {len(tracks_df)}')
print(f'Álbumes: {tracks_df["album"].nunique()}')
print(f'Período: {tracks_df["year"].min()} – {tracks_df["year"].max()}')
print()
print('Distribución por álbum:')
print(tracks_df.groupby('album')['name'].count().reindex(ALBUM_ORDER).to_string())

In [ ]:
# Timeline de álbumes de estudio
albums_for_timeline = tracks_df.groupby(['album', 'year'])['name'].count().reset_index()
albums_for_timeline.columns = ['album', 'year', 'total_tracks']

fig_timeline = px.scatter(
    albums_for_timeline.sort_values('year'),
    x='year',
    y='album',
    size='total_tracks',
    color='total_tracks',
    hover_name='album',
    hover_data={'total_tracks': True, 'year': True},
    title='Discografía de estudio — Radiohead',
    color_continuous_scale='Viridis',
    size_max=45,
    labels={'year': 'Año', 'album': 'Álbum', 'total_tracks': 'Pistas'},
)
fig_timeline.update_layout(
    height=400,
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False,
)
fig_timeline.show()

## 5. Estadísticas Descriptivas de Audio Features

In [ ]:
# Estadísticas globales
stats = tracks_df[FEAT_COLS].describe().round(4)
stats.index = ['Conteo', 'Media', 'Desv. estándar', 'Mínimo',
               'P25', 'Mediana', 'P75', 'Máximo']
print('Estadísticas descriptivas de Audio Features — Radiohead (discografía completa):')
display(stats)

In [ ]:
# Media por álbum
album_means = (
    tracks_df.groupby('album')[FEAT_COLS]
    .mean()
    .round(4)
    .reindex(ALBUM_ORDER)
)
print('Media de audio features por álbum:')
display(album_means)

In [ ]:
# Barras de medias globales
global_means = tracks_df[FEAT_COLS].mean().sort_values(ascending=False)

fig_means = px.bar(
    x=global_means.index,
    y=global_means.values,
    color=global_means.values,
    color_continuous_scale='Viridis',
    title='Media global de audio features — Radiohead',
    labels={'x': 'Feature', 'y': 'Valor medio', 'color': 'Valor'},
)
fig_means.update_layout(showlegend=False, coloraxis_showscale=False, height=380)
fig_means.show()

## 6. Evolución Temporal del Sonido

In [ ]:
# Calcular medias por álbum (orden cronológico)
evo_df = (
    tracks_df.groupby(['year', 'album'])[FEAT_COLS]
    .mean()
    .reset_index()
    .sort_values('year')
)

features_to_plot = ['energy', 'valence', 'acousticness', 'instrumentalness', 'danceability']
palette = px.colors.qualitative.Plotly

fig_evo = go.Figure()
for i, feat in enumerate(features_to_plot):
    fig_evo.add_trace(go.Scatter(
        x=evo_df['album'],
        y=evo_df[feat],
        mode='lines+markers',
        name=feat.title(),
        line={'color': palette[i % len(palette)], 'width': 2},
        marker={'size': 9},
        hovertemplate=f'<b>{feat.title()}</b><br>%{{x}}<br>%{{y:.3f}}<extra></extra>',
    ))

fig_evo.update_layout(
    title='Evolución de audio features por álbum de estudio — Radiohead',
    xaxis={'categoryorder': 'array', 'categoryarray': ALBUM_ORDER, 'tickangle': -25},
    yaxis={'range': [0, 1], 'title': 'Valor medio'},
    legend={'orientation': 'h', 'y': -0.22},
    height=460,
)
fig_evo.show()

In [ ]:
# Gráfico de área apilada — energy y acousticness a lo largo de la discografía
fig_area = go.Figure()

fig_area.add_trace(go.Scatter(
    x=evo_df['album'], y=evo_df['energy'],
    name='Energy', fill='tozeroy',
    line={'color': '#1DB954', 'width': 2},
    fillcolor='rgba(29,185,84,0.25)',
))
fig_area.add_trace(go.Scatter(
    x=evo_df['album'], y=evo_df['acousticness'],
    name='Acousticness', fill='tozeroy',
    line={'color': '#1E90FF', 'width': 2},
    fillcolor='rgba(30,144,255,0.25)',
))
fig_area.add_trace(go.Scatter(
    x=evo_df['album'], y=evo_df['instrumentalness'],
    name='Instrumentalness', fill='tozeroy',
    line={'color': '#FF6B6B', 'width': 2},
    fillcolor='rgba(255,107,107,0.25)',
))

fig_area.update_layout(
    title='Transición Energy → Acousticness → Instrumentalness a lo largo de la carrera',
    xaxis={'categoryorder': 'array', 'categoryarray': ALBUM_ORDER, 'tickangle': -25},
    yaxis={'range': [0, 1]},
    legend={'orientation': 'h'},
    height=400,
)
fig_area.show()

## 7. Scatter Plot — Energy vs Valence

In [ ]:
fig_scatter = px.scatter(
    tracks_df,
    x='energy',
    y='valence',
    color='album',
    hover_name='name',
    hover_data={'album': True, 'year': True, 'energy': ':.3f', 'valence': ':.3f'},
    title='Energy vs Valence por pista y álbum — Radiohead',
    labels={'energy': 'Energía', 'valence': 'Valencia (positividad)', 'album': 'Álbum'},
    color_discrete_sequence=px.colors.qualitative.Set2,
)

# Cuadrantes de referencia
fig_scatter.add_hline(y=0.5, line_dash='dash', line_color='gray', opacity=0.4)
fig_scatter.add_vline(x=0.5, line_dash='dash', line_color='gray', opacity=0.4)

fig_scatter.update_layout(height=500)
fig_scatter.show()

In [ ]:
# Scatter: Danceability vs Instrumentalness (colorizado por año)
fig_s2 = px.scatter(
    tracks_df,
    x='danceability',
    y='instrumentalness',
    color='year',
    size='duration_min',
    hover_name='name',
    hover_data={'album': True, 'year': True},
    title='Danceability vs Instrumentalness (tamaño = duración)',
    color_continuous_scale='Viridis',
    labels={'danceability': 'Bailabilidad', 'instrumentalness': 'Instrumental'},
)
fig_s2.update_layout(height=460)
fig_s2.show()

## 8. Heatmap — Features × Álbum

In [ ]:
# Heatmap features × álbum
hm_data = album_means.copy()

fig_heat = px.imshow(
    hm_data.T,
    color_continuous_scale='RdYlGn',
    aspect='auto',
    title='Heatmap de audio features por álbum (verde = alto, rojo = bajo)',
    labels={'x': 'Álbum', 'y': 'Feature', 'color': 'Valor medio'},
    zmin=0, zmax=1,
)
fig_heat.update_xaxes(tickangle=-25)
fig_heat.update_layout(height=380)
fig_heat.show()

In [ ]:
# Heatmap pistas × features para un álbum específico
SELECTED_ALBUM = 'OK Computer'

alb_data = (
    tracks_df[tracks_df['album'] == SELECTED_ALBUM]
    .sort_values('track_number')
    .set_index('name')[['energy', 'danceability', 'valence', 'acousticness', 'instrumentalness']]
)

fig_heat2 = px.imshow(
    alb_data.T,
    color_continuous_scale='RdYlGn',
    title=f'Audio features por pista — {SELECTED_ALBUM}',
    labels={'x': 'Canción', 'y': 'Feature', 'color': 'Valor'},
    zmin=0, zmax=1,
)
fig_heat2.update_xaxes(tickangle=-40)
fig_heat2.update_layout(height=320)
fig_heat2.show()

## 9. Radar Chart — Perfil Sonoro

In [ ]:
# Radar chart comparando todos los álbumes
radar_feats = ['energy', 'danceability', 'valence', 'acousticness', 'instrumentalness', 'liveness']
palette = px.colors.qualitative.Set2

fig_radar = go.Figure()
for i, alb in enumerate(ALBUM_ORDER):
    if alb not in album_means.index:
        continue
    vals = album_means.loc[alb, radar_feats].tolist()
    fig_radar.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=radar_feats + [radar_feats[0]],
        mode='lines+markers',
        name=alb,
        line={'color': palette[i % len(palette)], 'width': 2},
        fill='toself',
        fillcolor=palette[i % len(palette)],
        opacity=0.15,
    ))

fig_radar.update_layout(
    polar={'radialaxis': {'visible': True, 'range': [0, 1]}},
    title='Perfil de audio features por álbum — Radiohead',
    legend={'orientation': 'h'},
    height=560,
)
fig_radar.show()

## 10. Análisis de Duración

In [ ]:
# Distribución de duración por álbum (boxplot)
ordered_df = tracks_df.copy()
ordered_df['album'] = pd.Categorical(ordered_df['album'], categories=ALBUM_ORDER, ordered=True)
ordered_df = ordered_df.sort_values('album')

fig_dur = px.box(
    ordered_df,
    x='album',
    y='duration_min',
    color='album',
    points='all',
    hover_name='name',
    title='Distribución de duración por álbum — Radiohead',
    labels={'album': 'Álbum', 'duration_min': 'Duración (min)'},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig_dur.update_layout(xaxis_tickangle=-30, showlegend=False, height=420)
fig_dur.show()

In [ ]:
# Duración total por álbum
dur_total = (
    ordered_df.groupby('album')['duration_min']
    .sum()
    .reset_index()
    .sort_values('album')
)

fig_dur2 = px.bar(
    dur_total,
    x='album',
    y='duration_min',
    color='duration_min',
    color_continuous_scale='Blues',
    text='duration_min',
    title='Duración total por álbum (minutos)',
    labels={'album': 'Álbum', 'duration_min': 'Duración total (min)'},
)
fig_dur2.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig_dur2.update_layout(
    xaxis_tickangle=-30, showlegend=False, coloraxis_showscale=False, height=400
)
fig_dur2.show()

## 11. Exportar JSON para la Web App

Exportamos los datos procesados a `data/` para que la web app Streamlit los lea de forma offline.

In [ ]:
# ── Preparar datos de álbumes ─────────────────────────────────────────────────
albums_out = []
for i, alb_name in enumerate(ALBUM_ORDER):
    alb_tracks = tracks_df[tracks_df['album'] == alb_name]
    if alb_tracks.empty:
        continue
    means = {f: round(float(alb_tracks[f].mean()), 4) for f in FEAT_COLS}
    year = int(alb_tracks['year'].iloc[0])
    release_date = str(alb_tracks['release_date'].iloc[0].date())
    albums_out.append({
        'album_id': f'rh_album_{i+1:02d}',
        'name': alb_name,
        'release_date': release_date,
        'year': year,
        'total_tracks': len(alb_tracks),
        'album_type': 'album',
        **means,
        'mean_loudness': round(float(alb_tracks['loudness'].mean()), 3),
        'mean_tempo': round(float(alb_tracks['tempo'].mean()), 3),
    })

# ── Preparar tracks ───────────────────────────────────────────────────────────
tracks_out = tracks_df.copy()
tracks_out['release_date'] = tracks_out['release_date'].astype(str)

# ── Preparar stats ────────────────────────────────────────────────────────────
def _stats_of(vals):
    n = len(vals)
    mean = sum(vals) / n
    std = math.sqrt(sum((v - mean) ** 2 for v in vals) / n)
    return {'mean': round(mean, 4), 'std': round(std, 4),
            'min': round(min(vals), 4), 'max': round(max(vals), 4)}

feat_stats = {f: _stats_of(tracks_df[f].tolist()) for f in FEAT_COLS}

stats_out = {
    'artist': 'Radiohead',
    'total_tracks': len(tracks_df),
    'total_albums': len(albums_out),
    'albums': ALBUM_ORDER,
    'year_range': [int(tracks_df['year'].min()), int(tracks_df['year'].max())],
    'features': feat_stats,
}

# ── Guardar JSON ──────────────────────────────────────────────────────────────
DATA_DIR.mkdir(parents=True, exist_ok=True)

(DATA_DIR / 'radiohead_albums.json').write_text(
    json.dumps(albums_out, indent=2), encoding='utf-8'
)
(DATA_DIR / 'radiohead_tracks.json').write_text(
    json.dumps(tracks_out.to_dict('records'), indent=2), encoding='utf-8'
)
(DATA_DIR / 'radiohead_stats.json').write_text(
    json.dumps(stats_out, indent=2), encoding='utf-8'
)

print('✅ Archivos JSON exportados a data/')
print(f'   radiohead_albums.json  → {len(albums_out)} álbumes')
print(f'   radiohead_tracks.json  → {len(tracks_df)} pistas')
print(f'   radiohead_stats.json   → stats globales')

## Conclusiones

### Requisito 1 — Filtrado de álbumes de estudio

El algoritmo `filter_studio_albums()` identificó correctamente los 7 álbumes de estudio de Radiohead  
a partir de ~28 ítems en la discografía (incluyendo remasters, singles, EPs y compilaciones).

Las cuatro reglas aplicadas son **necesarias y suficientes**:
- Sin la regla de tipo, los singles pasarían el filtro.
- Sin la exclusión por keywords, los remasters y álbumes en vivo quedarían incluidos.
- Sin el mínimo de pistas, algunos EPs mal clasificados pasarían como álbumes.
- Sin la deduplicación, versiones remasterizadas del mismo álbum aparecerían duplicadas.

### Requisito 2 — Audio Features

Los datos revelan la **evolución estilística clara** de Radiohead:

| Feature | Tendencia | Interpretación |
|---------|-----------|----------------|
| **Energy** | ↘ decreciente | De rock alternativo a sonidos más contenidos y atmósferos |
| **Acousticness** | ↗ creciente hacia 2016 | Mayor uso de instrumentos acústicos en la etapa madura |
| **Instrumentalness** | Pico en Kid A/Amnesiac | Fase experimental electrónica (2000–2001) |
| **Valence** | Consistentemente baja | Carácter melancólico y sombrío a lo largo de toda la carrera |
| **Danceability** | Moderada, ligero pico en Kid A | Ritmos electrónicos aumentan la bailabilidad sin ser dance |

**Conclusión:** Los audio features de Spotify capturan con precisión la transición del rock  
alternativo de los 90s a la electrónica experimental y el post-rock de la discografía tardía.